# One ExoMiner++ 2.0 sample: inputs and output

Reads **one random TCE** from the normalized round-0 data and shows exactly what the model gets.
- `data_sample`: the raw TFRecord example.
- `input_example`: all input features, then split into `input_light_curves`, `input_images`, `input_scalars`.
- `output_sample`: the label (what the model is trained to predict).

**How features are read** mirrors NASA's input pipeline, `src/utils/utils_dataio.py` (`_prepare_data_fields`, lines ~268-274, and `parse_and_reshape`, lines ~344-350):
- `unfolded` features are stored as serialized tensors.
- Other N-D features are stored flattened and reshaped with `dim`.
- Scalars are stored as-is.

**Which features and how they're grouped** come from the run's own config (`config_cv.yaml`: `features_set` + the model config's branches), so this is exactly what the trained baseline sees.

**Run** (in WSL): `conda activate exoplnt_dl`, then `pip install notebook` if needed, then `jupyter notebook --no-browser`, and open the printed URL.

In [ ]:
import random
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
import tensorflow as tf
import matplotlib.pyplot as plt

## Paths and config

In [ ]:
# normalized data for CV round 0 (inside WSL's filesystem)
NORM_DATA_DIR = Path('/home/fhard/exominer_data/eval_normalized/cv_iter_0/norm_data')
# merged config written by setup_cv_iter.py for the baseline run: features, dims, label map, model branches
RUN_CONFIG_FP = Path('/mnt/c/Users/fhard/GitHub/ExoMiner/exominer_data/experiments/baseline/cv_iter_0/models/model0/config_cv.yaml')
with open(RUN_CONFIG_FP) as f:
    run_config = yaml.unsafe_load(f)  # unsafe_load: file contains Path objects
features_set = run_config['features_set']  # feature name -> {'dim': [...], 'dtype': ...}
model_config = run_config['config']  # architecture: which features go to which branch
label_map = run_config['label_map']  # label string -> 0/1
print(f'{len(features_set)} input features; label map: {label_map}')

## Read a sample: `data_sample`

In [ ]:
shard_fps = sorted(NORM_DATA_DIR.glob('shard-fold-*.tfrecord'))
shard_fp = random.choice(shard_fps)
record_i = random.randint(0, 299)  # shards hold ~500+ examples; stay safely inside
raw_record = next(iter(tf.data.TFRecordDataset(str(shard_fp)).skip(record_i).take(1)))
data_sample = tf.train.Example.FromString(raw_record.numpy())  # protobuf: feature name -> list of values
print(f'shard: {shard_fp.name}, record {record_i}')
print(f'{len(data_sample.features.feature)} fields stored in this example (inputs + metadata)')
def get_field(name):  # read a metadata field (string or number)
    feat = data_sample.features.feature[name]
    kind = feat.WhichOneof('kind')
    vals = getattr(feat, kind).value
    return vals[0].decode() if kind == 'bytes_list' else vals[0]
for name in ['uid', 'target_id', 'tce_plnt_num', 'label', 'obs_type', 'tce_period', 'tce_depth']:
    if name in data_sample.features.feature:
        print(f'  {name}: {get_field(name)}')

## Input section: `input_example`

In [ ]:
def parse_feature(name, info):
    """Same rules as NASA's parse_and_reshape in src/utils/utils_dataio.py."""
    feat = data_sample.features.feature[name]
    dim = info['dim']
    if len(dim) > 1 and dim[-1] > 1 and 'unfolded' in name:  # stored as a serialized tensor
        tensor = tf.io.parse_tensor(feat.bytes_list.value[0], out_type=tf.float32)
        return tf.reshape(tensor, dim).numpy()
    return np.array(feat.float_list.value, dtype=np.float32).reshape(dim)  # flattened -> original shape
input_example = {name: parse_feature(name, info) for name, info in features_set.items()}
print(f'input_example: {len(input_example)} features')

### Split by the model's own branches
- **Light curves:** the 1-D convolution branches (`conv_branches`); where the global-context layer will go.
- **Images:** the difference-image branch.
- **Scalars:** everything else (stellar + signal-fit numbers, plus per-branch summary scalars).

In [ ]:
light_curve_names = [v for branch in model_config['conv_branches'].values() for v in branch['views']]
image_names = model_config['diff_img_branch']['imgs'] + model_config['diff_img_branch']['imgs_scalars']
input_light_curves = {n: input_example[n] for n in light_curve_names}
input_images = {n: input_example[n] for n in image_names}
input_scalars = {n: v for n, v in input_example.items() if n not in input_light_curves and n not in input_images}
def summary(group):
    return pd.DataFrame([{'feature': n, 'shape': v.shape, 'min': v.min(), 'max': v.max(), 'mean': v.mean()} for n, v in group.items()])
for title, group in [('LIGHT CURVES', input_light_curves), ('IMAGES', input_images), ('SCALARS', input_scalars)]:
    print(f'\n{title}: {len(group)} features')
    display(summary(group))

### Light curves (one panel per view; `_var` = per-bin variance; unfolded = individual transits)

In [ ]:
names = [n for n in light_curve_names if not n.endswith('_var')]
fig, axes = plt.subplots(len(names), 1, figsize=(10, 2.2 * len(names)))
for ax, n in zip(axes, names):
    v = input_light_curves[n]
    if v.ndim == 2 and v.shape[1] > 1:  # unfolded: (transits, bins) -> image
        ax.imshow(v, aspect='auto', cmap='viridis'); ax.set_ylabel('transit #')
    else:
        ax.plot(v.squeeze(), lw=1)
    ax.set_title(f'{n}  {v.shape}', fontsize=9)
plt.tight_layout(); plt.show()

### Images (first of the 5 stacked images per type; 55×55 pixels around the target)

In [ ]:
img_names = model_config['diff_img_branch']['imgs']
fig, axes = plt.subplots(1, len(img_names), figsize=(4 * len(img_names), 4))
for ax, n in zip(axes, img_names):
    im = ax.imshow(input_images[n][0], cmap='viridis'); ax.set_title(f'{n}\n{input_images[n].shape}', fontsize=8)
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()
print('quality (one value per stacked image):', input_images['quality'])

### Scalars

In [ ]:
pd.Series({n: float(v.squeeze()) for n, v in input_scalars.items()}, name='value').to_frame()

## Output section: `output_sample`
The model outputs one probability: planet (1) vs. not planet (0). The training target is the TCE's label mapped through `label_map` (KP, CP → 1; FP, EB, NTP, BD → 0).

In [ ]:
label = get_field(run_config['label_field_name'])
output_sample = label_map[label]
print(f'label: {label} -> output_sample (target): {output_sample}')